In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.database import Database
from birddog.database_updater import normalize_url

2026-08-14 15:19:21,269 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-08-14 15:19:21,385 [INFO] Translation is enabled. Using GCP translator
2026-08-14 15:19:21,386 [INFO] Using Google Cloud translation API
2026-08-14 15:19:21,386 [INFO] GoogleCloudTranslator using REST API


In [4]:
db = Database()

2026-08-14 15:19:23,180 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-14 15:19:23,337 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     6.45    39.00       0.00           24


In [7]:
mo_docs = []
cursor = None
while True:
    batch, cursor = db.scan(
        "Documents", 
        view_name="BD:Multi-Owner", 
        fields=["url", "owning_pages"],
        limit=1000,
    )
    mo_docs.extend(batch)
    break

In [8]:
mo_docs[0]

{'Id': 489673,
 'url': 'https://commons.wikimedia.org/wiki/File:ДАКО_П-23._Описи_1_та_2._Березанський_районний_комітет_КП_України,_смт._Березань,_Київська_область.pdf',
 'owning_pages': [{'Id': 1077877, 'title': 'ДАКО/П-23'},
  {'Id': 1077879, 'title': 'ДАКО/П-23/1'},
  {'Id': 1077880, 'title': 'ДАКО/П-23/2'}]}

In [9]:
len(mo_docs)

1000

In [10]:
def get_owner_labels(db, doc_rec):
    owner_ids = [p["Id"] for p in doc_rec.get("owning_pages", [])]
    doc_rec["owning_pages"] = db.read("Pages", owner_ids, fields=["title", "label"])
    return doc_rec

In [12]:
get_owner_labels(db, mo_docs[0])

2026-08-14 15:26:15,264 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   23.00     0.01    39.00       0.00           24


{'Id': 489673,
 'url': 'https://commons.wikimedia.org/wiki/File:ДАКО_П-23._Описи_1_та_2._Березанський_районний_комітет_КП_України,_смт._Березань,_Київська_область.pdf',
 'owning_pages': [{'Id': 1077877, 'title': 'ДАКО/П-23', 'label': 'DAKO/P-23'},
  {'Id': 1077879, 'title': 'ДАКО/П-23/1', 'label': 'DAKO/P-23/1'},
  {'Id': 1077880, 'title': 'ДАКО/П-23/2', 'label': 'DAKO/P-23/2'}]}